# Title: Feature Engineering
### Author: Kolbe Sussman

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import joblib


In [2]:
# Paths
FEATURES_CSV = "../data/processed/features.csv"
OUTPUT_DIR = "outputs/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load features
print("Loading features...")
df = pd.read_csv(FEATURES_CSV)

Loading features...


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/features.csv'

In [ ]:
# Features & target
TARGET = "label"
EXCLUDE_COLS = ['author_1', 'author_2', TARGET]
X = df.drop(columns=EXCLUDE_COLS)
y = df[TARGET]

# Split into train/test
print("Splitting data into train/test...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
# Model 1: Logistic Regression
# -------------------------
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, n_jobs=-1)
lr_model.fit(X_train, y_train)

# Evaluate
y_pred_prob = lr_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_prob)
print(f"Logistic Regression AUC: {auc:.4f}")

# Save model
# joblib.dump(lr_model, os.path.join(OUTPUT_DIR, "logistic_regression.pkl"))

In [ ]:
# Model 2: Random Forest
# -------------------------
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_prob)
print(f"Random Forest AUC: {auc:.4f}")

# Save model
# joblib.dump(rf_model, os.path.join(OUTPUT_DIR, "random_forest.pkl"))

# Optional: feature importance
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.to_csv(os.path.join(OUTPUT_DIR, "rf_feature_importances.csv"))

print("Modeling complete. Models saved to outputs/models.")